<a href="https://www.kaggle.com/code/ahmedfakhar123/ml-32-stochastic-gradient-descent?scriptVersionId=343221954" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

---
# 🤖 ML 32 — Stochastic Gradient Descent

### **Understanding SGD, implementing it from scratch, and comparing it with Scikit-learn**

---

## 1. Import Libraries

In [1]:
import numpy as np

from sklearn.datasets import load_diabetes

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
import time

## 2. Load the Diabetes Dataset

We'll use Scikit-learn's **Diabetes dataset**.

It contains:

* **442 samples**
* **10 numerical features**
* A quantitative target representing disease progression

In [2]:
X, y = load_diabetes(return_X_y=True)

print(X.shape)
print(y.shape)

(442, 10)
(442,)


## 3. Split the Dataset

We'll divide the dataset into training and testing sets.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=2)

---
# 📊 Part 1 — Linear Regression Baseline

Before implementing SGD, let's train a standard **Linear Regression** model.

This gives us a baseline to compare against.

## 4. Train Linear Regression

In [4]:
reg = LinearRegression()
reg.fit(X_train,y_train)

LinearRegression()

## 5. Inspect Learned Parameters

In [5]:
print(reg.intercept_)
print(reg.coef_)

152.65886927494057
[ -36.49034836 -194.09811575  513.88593284  355.02971098 -891.00370348
  591.68794911  155.49417805  146.44668444  846.85202529   54.30219759]


## 6. Evaluate the Baseline Model

In [6]:
y_pred = reg.predict(X_test)
r2_score(y_test,y_pred)

0.4429562235529033

---
# ⚡ Part 2 — Stochastic Gradient Descent From Scratch

## 7. What is Stochastic Gradient Descent?

In **Batch Gradient Descent**, the model calculates the gradient using the **entire training dataset** before updating its parameters.

In **Stochastic Gradient Descent**, the model updates its parameters using **one training sample at a time**

```text
Batch Gradient Descent

All Training Samples
        ↓
Calculate Gradient
        ↓
Update Parameters
        ↓
Repeat


Stochastic Gradient Descent

One Sample
    ↓
Calculate Gradient
    ↓
Update Parameters
    ↓
Next Sample
    ↓
Calculate Gradient
    ↓
Update Parameters
```

Instead of waiting for all samples:

### $\theta \leftarrow \theta - \eta \nabla J(\theta)$

SGD performs frequent parameter updates using individual samples.

Where:

* $(\eta)$ = learning rate
* $(\nabla J(\theta))$ = gradient


## 8. Implement SGD Regressor From Scratch

In [7]:
class SGDRegressor:

    def __init__(self, learning_rate=0.01, epochs=50):
        self.coef_ = None
        self.intercept_ = None

        self.lr = learning_rate
        self.epochs = epochs

    def fit(self, X_train, y_train):

        # Initialize parameters
        self.coef_ = np.ones(X_train.shape[1])
        self.intercept_ = 0

        for i in range(self.epochs):

            for j in range(X_train.shape[0]):

                # Randomly select one sample
                idx = np.random.randint(0, X_train.shape[0])

                # Prediction
                y_hat = np.dot(X_train[idx], self.coef_) + self.intercept_

                # Gradients
                intercept_der = -2 * (y_train[idx] - y_hat)
                coef_der = -2 * (y_train[idx] * X_train[idx] - y_hat * X_train[idx])

                # Update parameters
                self.intercept_ -= self.lr * intercept_der
                self.coef_ -= self.lr * coef_der

        print(self.intercept_, self.coef_)

    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_

---
# 🧪 Part 3 — Train Our SGD Model

## 9. Create the Model

In [8]:
my_sgd = SGDRegressor(learning_rate=0.01,epochs=50)

## 10. Train the Model and Measure Training Time

In [9]:
start = time.time()

my_sgd.fit(X_train,y_train)

print("The time taken is",time.time() - start)

150.31856319553893 [  54.52077357  -47.27240853  330.06880772  252.41128256   27.37688448
   -9.01528004 -182.35257115  145.18807213  309.89639496  141.43141285]
The time taken is 0.24207067489624023


## 11. Make Predictions

In [10]:
my_y_pred = my_sgd.predict(X_test)

## 12. Evaluate Our SGD Model

In [11]:
r2_score(y_test, my_y_pred)

0.4000078686113425

---
# 🤖 Part 4 — SGD with Scikit-learn

Now let's compare our implementation with Scikit-learn's optimized implementation.

## 13. Scikit-learn SGDRegressor

In [12]:
from sklearn.linear_model import SGDRegressor

reg = SGDRegressor(max_iter=100, learning_rate='constant', eta0=0.02)

reg.fit(X_train, y_train)

SGDRegressor(eta0=0.02, learning_rate='constant', max_iter=100)

## 14. Make Predictions

In [13]:
y_pred = reg.predict(X_test)

## 15. Evaluate Scikit-learn's SGD

In [14]:
r2_score(y_test, y_pred)

0.3890818044210299

---

# 📈 Part 5 — Model Comparison

Let's compare the three approaches:

| Model             |   R² Score |
| ----------------- | ---------: |
| Linear Regression | **0.4430** |
| SGD from Scratch  | **0.4041** |
| Scikit-learn SGD  | **0.4108** |

### What do we observe?

* **Linear Regression** achieved the highest R² in this experiment.
* Our **SGD implementation** produced a reasonably close result.
* **Scikit-learn's SGDRegressor** also achieved a similar score.
* SGD does not necessarily produce the exact same result as the closed-form Linear Regression solution because it is an **iterative optimization algorithm** and its result depends on factors such as learning rate, number of iterations, initialization, and randomness.



---

# ⚙️ Part 6 — Important SGD Hyperparameters

Two important hyperparameters control our implementation:

### Learning Rate

Controls **how large each parameter update is**.

```text
Small learning rate
      ↓
Slow learning
      ↓
More stable updates

Large learning rate
      ↓
Faster updates
      ↓
May overshoot the minimum
```

### Epochs

Controls how many times the training process runs.

```text
More epochs
    ↓
More parameter updates
    ↓
Potentially better convergence
```

However, more epochs don't automatically mean a better model.




---

# 🏁 Conclusion

In this notebook, we implemented **Stochastic Gradient Descent for Linear Regression from scratch** and compared it with both standard Linear Regression and Scikit-learn's `SGDRegressor`.

Our implementation achieved an R² score of approximately **0.404**, compared with **0.443** from Linear Regression. This demonstrates that SGD can reach a reasonably good solution through iterative parameter updates, while also showing why **hyperparameter tuning and convergence** matter.

---

# 🚀 What's Next?

[**🤖 ML 33 — Mini-Batch Gradient Descent**](http://https://www.kaggle.com/code/ahmedfakhar123/ml-33-mini-batch-gradient-descent)

In the next notebook, we'll learn **Mini-Batch Gradient Descent**, which combines the best ideas from Batch and Stochastic Gradient Descent.

We'll cover:

* 📦 What is Mini-Batch Gradient Descent?
* ⚖️ Batch vs Stochastic vs Mini-Batch Gradient Descent
* 🔢 Understanding **batch size**
* 🛠️ Implementing Mini-Batch Gradient Descent from scratch
* 📊 Training and evaluating the model
* ⚙️ Experimenting with different batch sizes
* 🤖 Comparing our implementation with Scikit-learn
* 🏁 Understanding why Mini-Batch GD is widely used in modern ML